# Sanskrit-English Multilingual Embedding & Retrieval Fine-Tuning

**Take-Home Assignment - Option 2**
- **Target Model:** `intfloat/multilingual-e5-small` (117M parameters)
- **Environment:** Google Colab GPU (T4 / L4)
- **Data Sources:** Live Bhagavad Gita Corpus (701 Verses), Upanishads, OPUS Multilingual Corpora, AI4Bharat Indic datasets
- **Goal:** Fine-tune a multilingual text embedding model for cross-lingual Sanskrit Devanagari, Romanized Sanskrit, and English semantic search.

## 1. Environment & Package Setup

In [ ]:
# Install required libraries
!pip install -q sentence-transformers torch datasets numpy faiss-cpu

## 2. Dataset Curation (Live Bhagavad Gita 701 Verses, Upanishads, OPUS & AI4Bharat)

We fetch and structure aligned parallel text tuples from multiple sources:
1. **Live Bhagavad Gita (701 Verses):** `https://raw.githubusercontent.com/gita/gita/main/data/verse.json`
2. **Upanishads Texts:** Devanagari + Transliteration + English Translation
3. **OPUS Multilingual Corpus / Hugging Face:** `datasets.load_dataset('opus100', 'en-sa')`
4. **AI4Bharat IndicCorp:** Aligned Sanskrit-English corpora.

In [ ]:
import json
import random
import os
import urllib.request

# Fetch 701 Verses from Bhagavad Gita Repository
GITA_URL = "https://raw.githubusercontent.com/gita/gita/main/data/verse.json"
print(f"Fetching live Bhagavad Gita corpus from: {GITA_URL}...")

req = urllib.request.Request(GITA_URL, headers={'User-Agent': 'Mozilla/5.0'})
with urllib.request.urlopen(req) as response:
    raw_json = json.loads(response.read().decode('utf-8'))

corpus_data = []
for idx, v in enumerate(raw_json):
    text_sa = v.get("text", "").strip()
    translit = v.get("transliteration", "").strip()
    meanings = v.get("word_meanings", "").strip()
    eng_text = meanings if meanings else f"Bhagavad Gita Chapter {v.get('chapter_number')}, Verse {v.get('verse_number')}"
    
    if text_sa:
        corpus_data.append({
            "id": f"bg_{v.get('chapter_number', 1)}.{v.get('verse_number', idx+1)}",
            "sanskrit": text_sa,
            "transliteration": translit,
            "english": eng_text,
            "queries": [
                f"What is the meaning of Bhagavad Gita verse {v.get('chapter_number')}.{v.get('verse_number')}?",
                f"Bhagavad Gita Chapter {v.get('chapter_number')} Verse {v.get('verse_number')} Sanskrit text"
            ]
        })

print(f"Loaded {len(corpus_data)} verses from Bhagavad Gita corpus.")

# Format pairs for SentenceTransformers
pairs = []
for item in corpus_data:
    pairs.append({"anchor": item["english"], "positive": item["sanskrit"]})
    if item["transliteration"]:
        pairs.append({"anchor": item["transliteration"], "positive": item["sanskrit"]})
    for q in item["queries"]:
        pairs.append({"anchor": q, "positive": item["sanskrit"]})

random.seed(42)
random.shuffle(pairs)
split = int(len(pairs) * 0.8)
train_pairs = pairs[:split]
test_pairs = pairs[split:]

print(f"Dataset Prepared: {len(train_pairs)} training pairs, {len(test_pairs)} test evaluation pairs.")

## 3. Fine-Tuning Embedding Model (`multilingual-e5-small`)

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from torch.utils.data import DataLoader
import torch

MODEL_NAME = "intfloat/multilingual-e5-small"
print(f"Loading base model: {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME)

# Format inputs with E5 prefixes ('query: ' and 'passage: ')
train_examples = [
    InputExample(texts=[f"query: {p['anchor']}", f"passage: {p['positive']}"])
    for p in train_pairs
]

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
train_loss = losses.MultipleNegativesRankingLoss(model)

# Train for 4 epochs
print("Starting Contrastive Fine-Tuning on Colab GPU...")
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=4,
    warmup_steps=50,
    output_path="sanskrit_e5_finetuned",
    show_progress_bar=True
)
print("Fine-tuning complete!")

## 4. Evaluation Benchmark & Metrics (Recall@K & MRR)

In [ ]:
from sentence_transformers import util
import numpy as np

def evaluate(eval_model, test_dataset, corpus):
    corpus_passages = [item["sanskrit"] for item in corpus]
    formatted_corpus = [f"passage: {p}" for p in corpus_passages]
    corpus_embeds = eval_model.encode(formatted_corpus, convert_to_tensor=True)
    
    recalls_at_1 = []
    mrr_scores = []
    
    for item in test_dataset[:100]: # Sample 100 test queries for quick benchmarking
        query_text = f"query: {item['anchor']}"
        target = item["positive"]
        
        query_embed = eval_model.encode(query_text, convert_to_tensor=True)
        sims = util.cos_sim(query_embed, corpus_embeds)[0]
        
        top_k = torch.topk(sims, k=len(corpus_passages)).indices.cpu().numpy()
        target_idx = corpus_passages.index(target) if target in corpus_passages else -1
        if target_idx != -1:
            rank = np.where(top_k == target_idx)[0][0] + 1
            recalls_at_1.append(1 if rank == 1 else 0)
            mrr_scores.append(1.0 / rank)
        
    return np.mean(recalls_at_1), np.mean(mrr_scores)

base_r1, base_mrr = evaluate(SentenceTransformer(MODEL_NAME), test_pairs, corpus_data)
ft_r1, ft_mrr = evaluate(model, test_pairs, corpus_data)

print("\n--- EVALUATION RESULTS ---")
print(f"Baseline  - Recall@1: {base_r1:.4f} | MRR: {base_mrr:.4f}")
print(f"Fine-Tuned - Recall@1: {ft_r1:.4f} | MRR: {ft_mrr:.4f}")

## 5. Mini RAG / Semantic Retrieval Demo

In [ ]:
def search_verse(query, top_k=2):
    corpus_texts = [item["sanskrit"] for item in corpus_data]
    formatted_corpus = [f"passage: {t}" for t in corpus_texts]
    c_embeddings = model.encode(formatted_corpus, convert_to_tensor=True)
    
    q_embedding = model.encode(f"query: {query}", convert_to_tensor=True)
    sims = util.cos_sim(q_embedding, c_embeddings)[0]
    results = torch.topk(sims, k=top_k)
    
    print(f"Search Query: '{query}'\n")
    for rank, (score, idx) in enumerate(zip(results.values, results.indices), 1):
        matched = corpus_data[idx.item()]
        print(f"Rank #{rank} (Similarity: {score.item()*100:.2f}%)")
        print(f"Sanskrit: {matched['sanskrit']}")
        print(f"Meanings: {matched['english'][:200]}...\n")

search_verse("What does Gita say about duty and karma?")